In [1]:
%pip install transformers datasets evaluate peft torch scikit-learn datasets pandas

  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 5.9 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 5.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 5.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 3.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 2.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.1 MB/s  0:00:00 eta 0:00:01
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 4.2 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 5.6 MB/s  0:00:00
  Attempting uninstall: fsspec90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/24 [pyarrow]
    Found existing installation: fsspec 2026.1.0━━━━━━━━━━━━━━  4/24 [pyarrow]
    Uninstalling fsspec-2026.1.0:━━━━━━━━━━━━━━━━━━━━

In [1]:
import numpy as np
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pickle
import pandas as pd
from datasets import Dataset


# stop_words = set(stopwords.words('english'))
all_data = pickle.load(open("para_data.pkl", "rb"))

X_data = [para["text"] for para in all_data]
y_data = np.array([0 if para["label"] == "human" else 1 for para in all_data])

dataset = Dataset.from_dict({"text": X_data, "labels": y_data})

train_test = dataset.train_test_split(test_size=0.2, seed=45)
val_test = train_test["test"].train_test_split(test_size=0.5, seed=45)

/home/irishbumfuzzle/Coding/PreCog-task/.venv13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import torch.nn as nn
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

id2label = {0: "human", 1: "ai"}
label2id = {"human": 0, "ai": 1}

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1093.18it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
def tokenize(data):
    return tokenizer(
            data['text'],
            max_length=512,
            padding='max_length',
            return_attention_mask=True,
            truncation=True
        )

tokenized_datasets = {"train": train_test["train"].map(tokenize, batched=True),
                      "validation": val_test["train"].map(tokenize, batched=True),
                      "test": val_test["test"].map(tokenize, batched=True)}

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest"
)

Map: 100%|██████████| 326/326 [00:00<00:00, 5928.60 examples/s]


In [4]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")


for name, module in model.named_modules():
    if 'attn' in name or 'attention' in name:
        print(name)
        for sub_name, sub_module in module.named_modules():
            print(f"  - {sub_name}")


distilbert.transformer.layer.0.attention
  - 
  - q_lin
  - k_lin
  - v_lin
  - out_lin
  - dropout
distilbert.transformer.layer.0.attention.q_lin
  - 
distilbert.transformer.layer.0.attention.k_lin
  - 
distilbert.transformer.layer.0.attention.v_lin
  - 
distilbert.transformer.layer.0.attention.out_lin
  - 
distilbert.transformer.layer.0.attention.dropout
  - 
distilbert.transformer.layer.1.attention
  - 
  - q_lin
  - k_lin
  - v_lin
  - out_lin
  - dropout
distilbert.transformer.layer.1.attention.q_lin
  - 
distilbert.transformer.layer.1.attention.k_lin
  - 
distilbert.transformer.layer.1.attention.v_lin
  - 
distilbert.transformer.layer.1.attention.out_lin
  - 
distilbert.transformer.layer.1.attention.dropout
  - 
distilbert.transformer.layer.2.attention
  - 
  - q_lin
  - k_lin
  - v_lin
  - out_lin
  - dropout
distilbert.transformer.layer.2.attention.q_lin
  - 
distilbert.transformer.layer.2.attention.k_lin
  - 
distilbert.transformer.layer.2.attention.v_lin
  - 
distilbert.trans

In [5]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS
)


print_trainable_parameters(model)
lora_model = get_peft_model(model, config)
print_trainable_parameters(lora_model)


trainable params: 66955010 || all params: 66955010 || trainable%: 100.0
trainable params: 739586 || all params: 67694596 || trainable%: 1.0925332946813067


In [6]:
def metrics(eval_prediction):
    logits, labels = eval_prediction
    pred = np.argmax(logits, axis=1)
    accuracy = np.mean(pred == labels)
    return {"accuracy": accuracy}
    # auc_score = roc_auc_score(labels, pred)
    # return {"Val-AUC": auc_score}

training_args = TrainingArguments(
    output_dir="results",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    logging_steps=10,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=collator,
    compute_metrics=metrics,
)

print("\nStarting Training...")
trainer.train()

merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")


Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.004602,0.001624,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.90it/s]


('finetuned_model/tokenizer_config.json', 'finetuned_model/tokenizer.json')

In [7]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, roc_auc_score

pred_output = trainer.predict(tokenized_datasets["test"])
logits = pred_output.predictions
labels = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds = np.argmax(logits, axis=1)

scores = {
    "accuracy": accuracy_score(labels, preds),
    "precision": precision_score(labels, preds),
    "f1": f1_score(labels, preds),
    "recall": recall_score(labels, preds),
    "roc_auc": roc_auc_score(labels, probs[:, 1]),
}

scores

{'accuracy': 1.0, 'precision': 1.0, 'f1': 1.0, 'recall': 1.0, 'roc_auc': 1.0}